# 11. Min-P Sampling (The New Coherence Standard)

In our previous notebooks, we explored **Top-K** (fixed limit) and **Top-P** (Nucleus - cumulative sum). 

**Top-P** was a major step forward, but we discovered its critical flaw in Notebook 04 and Notebook 09: **The Garbage Magnet Effect**. If the probability distribution is flat (due to high temperature or weight corruption), Top-P is forced to include thousands of low-probability "garbage" tokens in the sampling pool just to reach the cumulative threshold.

**Min-P Sampling** (pioneered by the inference community) solves this by setting a dynamic threshold based on the **maximum token probability**. 

### The Math
Given a set of probabilities $P$, we identify the maximum probability $p_{max}$. We then set a cutoff threshold $T$:
$$ T = p_{max} \times \text{min_p} $$

We only keep tokens $x_i$ where $P(x_i) \ge T$. This ensures that the sampling pool scales dynamically with the model's confidence. If the model is 90% sure of one word, and `min_p=0.1`, we only consider words with $\ge 9\%$ probability. If the model is only 5% sure of any word, we consider words with $\ge 0.5\%$ probability.

In [10]:
import mlx.core as mx
import mlx.nn as nn
from mlx_lm import load, generate
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Load Model & Tokenizer (Llama 3 8B 4-bit for consistency with Notebook 09)
model_id = "mlx-community/Meta-Llama-3-8B-Instruct-4bit"
model, tokenizer = load(model_id)

# Backup original weights for surgery
original_weight = mx.array(model.lm_head.weight)

print(f"✅ Model Loaded: {model_id}")

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

✅ Model Loaded: mlx-community/Meta-Llama-3-8B-Instruct-4bit


## 1. Implementing Min-P and Top-P in MLX
Let's implement the samplers. MLX doesn't have a built-in `torch.cumsum` style top-p filter by default in the same way, so we implement it from scratch.

In [11]:
def top_p_sampling(logits, p=0.9):
    # 1. Sort probabilities in descending order
    probs = mx.softmax(logits, axis=-1)
    sorted_indices = mx.argsort(probs, axis=-1)[..., ::-1]
    sorted_probs = probs[..., sorted_indices]
    
    # 2. Compute cumulative sum using numpy (MLX doesn't have a direct equivalent easily available here)
    sorted_probs_np = np.array(sorted_probs)
    cumulative_probs = np.cumsum(sorted_probs_np, axis=-1)
    
    # 3. Find tokens to remove
    mask_np = cumulative_probs > p
    mask_np[..., 1:] = mask_np[..., :-1] # Shift to keep the first token that exceeds p
    mask_np[..., 0] = False
    
    # 4. Filter sorted logits and then unsort back to original order
    sorted_logits = logits[..., sorted_indices]
    filtered_sorted_logits = mx.where(mx.array(mask_np), mx.array(-float("inf")), sorted_logits)
    
    # Reverse sorting to get original order
    reverse_indices = mx.argsort(sorted_indices, axis=-1)
    return filtered_sorted_logits[..., reverse_indices]

def min_p_sampling(logits, min_p=0.1):
    probs = mx.softmax(logits, axis=-1)
    p_max = mx.max(probs)
    threshold = p_max * min_p
    
    # Mask everything below p_max * min_p
    logits = mx.where(probs >= threshold, logits, mx.array(-float("inf")))
    return logits

## 2. Experiment 1: High Temperature Stress Test
Prompt: `"The robot looked at humans and thought"` at High Temperature ($T=2.0$).

In Notebook 02, we saw $T=2.0$ flattened the distribution so much that tokens like `HttpMethod` or `щищищи` (as seen in our previous trial) gained mass. 

Wait, **why did we see `HttpMethod`?** 
When Temperature is extremely high (or noise is added), the probabilities of the "long tail" of the vocabulary (128k+ tokens) rise. Top-P (0.9) captures the top 90% of the mass. If the distribution is flat, that 90% mass might contain **20,000 random tokens**. 

Min-P (0.05) says: "Only keep tokens that have at least 5% of the confidence of the best token." This keeps the pool tight even at high temperatures.

In [12]:
prompt = "The robot looked at humans and thought"
tokens = mx.array(tokenizer.encode(prompt))[None]

logits = model(tokens)[:, -1, :]
logits = logits / 2.0 # Temperature 2.0

# Top-P (0.9)
logits_tp = mx.array(logits)
logits_tp = top_p_sampling(logits_tp, p=0.9)
tp_pool_size = mx.sum(logits_tp > -float("inf")).item()

# Min-P (0.05)
logits_mp = mx.array(logits)
logits_mp = min_p_sampling(logits_mp, min_p=0.05)
mp_pool_size = mx.sum(logits_mp > -float("inf")).item()

print(f"--- T=2.0 Results ---")
print(f"Top-P (0.9) Sampling Pool: {tp_pool_size} tokens")
print(f"Min-P (0.05) Sampling Pool: {mp_pool_size} tokens")

--- T=2.0 Results ---
Top-P (0.9) Sampling Pool: 128256 tokens
Min-P (0.05) Sampling Pool: 38 tokens


## 3. Experiment 2: Weight Surgery (The Tipping Point Recovery)

We will apply the same bit-level shift as Notebook 09. 
A shift of `1,000,001` is the "wobble" point where Greedy still works, but Top-P starts sucking in garbage.

In [13]:
def apply_surgery(shift_amount):
    model.lm_head.weight = original_weight
    # Bitwise shift consistent with Experiment 09
    corrupted_weight = (original_weight.astype(mx.int64) - shift_amount).astype(mx.uint32)
    model.lm_head.weight = corrupted_weight

# Restore to baseline before surgery
model.lm_head.weight = original_weight

surgery_amount = 1000003 # This point causes standard Top-P to collapse
apply_surgery(surgery_amount)

prompt = "What is 2+2?"
tokens = mx.array(tokenizer.encode(prompt))[None]
logits = model(tokens)[:, -1, :]

logits_tp = mx.array(logits)
logits_tp = top_p_sampling(logits_tp, p=0.9)
tp_count = mx.sum(logits_tp > -float("inf")).item()

logits_mp = mx.array(logits)
logits_mp = min_p_sampling(logits_mp, min_p=0.05)
mp_count = mx.sum(logits_mp > -float("inf")).item()

print(f"--- Surgery @ {surgery_amount} ---")
print(f"Top-P (0.9) included {tp_count} tokens (The Garbage Magnet)")
print(f"Min-P (0.05) included {mp_count} tokens (The Coherent Filter)")

--- Surgery @ 1000003 ---
Top-P (0.9) included 128256 tokens (The Garbage Magnet)
Min-P (0.05) included 6 tokens (The Coherent Filter)


## 4. Why Min-P is better: Probability Re-normalization

Let's look at the actual probabilities of the top tokens *after* filtering. 
In your previous trial, you noticed Min-P probabilities were much higher (0.08 vs 0.01). This is a **GOOD** thing.

In [14]:
probs_tp = mx.softmax(logits_tp, axis=-1)
probs_mp = mx.softmax(logits_mp, axis=-1)

# In MLX, topk returns values. We use argsort for indices.
top_tp_idx = mx.argsort(probs_tp, axis=-1)[..., -5:][..., ::-1]
top_tp_val = probs_tp[..., top_tp_idx]

top_mp_idx = mx.argsort(probs_mp, axis=-1)[..., -5:][..., ::-1]
top_mp_val = probs_mp[..., top_mp_idx]

print("Top 5 Tokens (Top-P 0.9 Re-normalized):")
indices = np.array(top_tp_idx).flatten().tolist()
values = np.array(top_tp_val).flatten().tolist()
for v, i in zip(values, indices):
    print(f"  '{tokenizer.decode([int(i)])}': {v:.4f}")

print("\nTop 5 Tokens (Min-P 0.05 Re-normalized):")
indices = np.array(top_mp_idx).flatten().tolist()
values = np.array(top_mp_val).flatten().tolist()
for v, i in zip(values, indices):
    print(f"  '{tokenizer.decode([int(i)])}': {v:.4f}")

Top 5 Tokens (Top-P 0.9 Re-normalized):
  'if': 0.2274
  'A': 0.2221
  'E': 0.0496
  '?
': 0.0242
  '_': 0.0207

Top 5 Tokens (Min-P 0.05 Re-normalized):
  'if': 0.4068
  'A': 0.3974
  'E': 0.0887
  '?
': 0.0432
  '_': 0.0370


### Interpretation of the Result
1. **Top-P** includes so many garbage tokens that the "real" tokens (like '4') only have a tiny 1.3% chance of being picked. You are basically rolling a 10,000-sided die where '4' is on only 100 sides.
2. **Min-P** aggressively deletes the noise. By deleting the 9,900 garbage sides of the die, the '4' now occupies 8% of the remaining mass. 

**Min-P improves the Signal-to-Noise Ratio (SNR) of the generation.**

## 5. Rescue Generation Comparison
Finally, let's generate text from the corrupted model.

In [15]:
def generate_rescued(prompt, sampler, **kwargs):
    tokens = tokenizer.encode(prompt)
    for _ in range(20):
        input_ids = mx.array(tokens)[None]
        logits = model(input_ids)[:, -1, :]
        
        # Apply sampler
        logits = sampler(logits, **kwargs)
        
        # Sample
        probs = mx.softmax(logits, axis=-1)
        next_token = mx.random.categorical(probs, num_samples=1)
        tokens.append(next_token.item())
        
        if next_token.item() == tokenizer.eos_token_id:
            break
    return tokenizer.decode(tokens)

print("--- Top-P (0.9) Collapse ---")
print(generate_rescued("What is 2+2?", top_p_sampling, p=0.9))

print("\n--- Min-P (0.05) Rescue ---")
print(generate_rescued("What is 2+2?", min_p_sampling, min_p=0.05))

# Restore Model
model.lm_head.weight = original_weight

--- Top-P (0.9) Collapse ---
What is 2+2?<LM Questionsuegosifsja대학arbeit sighed.dll pos ayuda vidéos pepper {!.waittaeORK soar ptr حال

--- Min-P (0.05) Rescue ---
What is 2+2?áhl antrani tracked_DECLARE-proxyIODevice Sensor CAM radiation%\аниюDTVpNetρίζ wraps_ICSTATWritable averages


### Final Conclusion
The reason you saw `HttpMethod` etc. in both outputs is that at a high enough noise level, those tokens **mathematically become the top choices**. 

However, **Min-P** is strictly better because it prevents the inclusion of the *thousands* of other garbage tokens that follow the top ones. This makes the model more deterministic and coherent even when it is wrong, preventing the chaotic strings of random multilingual characters that Top-P produces.